## Flow / Sequence

**Importing necessary libraries**

In [ ]:
import os
import base64
from typing import Dict
import requests
import time
import json
import logging
import fitz
from PIL import Image
from io import BytesIO

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

**API configuration**

In [ ]:
API_CONFIG = {
    "base_url": os.getenv("API_BASE_URL", "https://qa2.modelops.maximus.com/v1"),
    "base_oauth_endpoint":"https://qa-modelops.auth.us-west-2.amazoncognito.com",
    "endpoints": {
        "invoke_conv": "/models/bedrock",
        "status": "/jobs",
        "data": "/jobs",
        "oauth": "/oauth2/token",
    },
}

**Credentials**

In [ ]:
creds = {
    "client_id": os.getenv("CLIENT_ID", ""),
    "client_secret": os.getenv("CLIENT_SECRET", ""),
    "api_key": os.getenv("API_KEY", "")
}


**Function to Obtain OAuth Token**

In [ ]:
def get_oauth_token() -> str:
    """
    Get oauth token for calling ModelOps

    :return token: str - token required for modelOps authentication
    """
    oauth_url = f"{API_CONFIG['base_oauth_endpoint']}{API_CONFIG['endpoints']['oauth']}"
    client_auth = base64.b64encode(
        f"{creds['client_id']}:{creds['client_secret']}".encode()
    ).decode()
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {client_auth}",
    }
    request_body = {"grant_type": "client_credentials"}

    response = requests.post(oauth_url, headers=headers, data=request_body)
    response.raise_for_status()

    token_response = response.json()
    token = token_response.get("access_token")

    if not token:
        raise ValueError("No access_token in OAuth response")

    return token

**Function to Wait for Job Completion**

In [ ]:
def wait_for_job_completion(jobId: str, token: str) -> bool:
    """
    Polling for the job to be completed

    :param jobId: str - job id of the API call
    :param token: str - token to validate API calls of ModelOps

    :return : bool - True if the job is completed else False
    """

    url = f"{API_CONFIG['base_url']}{API_CONFIG['endpoints']['status']}/{jobId}"
    headers = {"Authorization": f"Bearer {token}"}
    total_wait_time = 120
    wait_interval = 10

    for _ in range(total_wait_time // wait_interval):
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        result = response.json()
        job_status = result.get("jobStatus")

        if job_status in ("Completed", "Served"):
            return True
        if job_status == "Error":
            logger.info("Error in calling the model")
            error_msg = result["errorMessage"]
            raise Exception(f"Error calling the model via ModelOps: {error_msg}")

        time.sleep(wait_interval)

    return False

**Function to Get Job Response**

In [ ]:
def get_response(jobId: str, token: str) -> Dict:
    """
    Get response from the ModelOps bedrock model

    :param jobId: str - job id of API call
    :param token: str - token to authenticate ModelOps API call

    :return result: Dict - result obtained from bedrock model
    """
    url = f"{API_CONFIG['base_url']}{API_CONFIG['endpoints']['data']}/{jobId}/data"
    headers = {"Authorization": f"Bearer {token}"}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    result = response.json()
    return result

**Function to encode the PDF to base64**

In [ ]:
def read_resume(pdf_path):
    with open(pdf_path, "rb") as file:
        content = file.read()
        base_64_encoded_data = base64.b64encode(content).decode('utf-8')
    return base_64_encoded_data

**Function to Invoke Bedrock**

In [ ]:
def invoke_llm(prompt: str, model_identifier: str, token: str, base_64_encoded_data) -> str:
    """
    Calling the bedrock model via ModelOps conv route

    :param prompt: str - prompt to be sent to the bedrock model
    :param model_identifier: str - id of the bedrock model
    :param token: str - oauth token to be used with ModelOps API

    :return job_id: str - job id of the API call
    """
    url = f"{API_CONFIG['base_url']}{API_CONFIG['endpoints']['invoke_conv']}/{model_identifier}"
    request_body = {
        "prompt": prompt,
        "attachments": {
            "type": "pdf",
            "base64EncodedData" : base_64_encoded_data,
            "documentName": "sample_resume"
        },
        "type": "converse"
        }
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
        "x-api-key": creds["api_key"],
    }

    response = requests.post(url, json=request_body, headers=headers)
    response.raise_for_status()

    result = response.json()
    return result.get("jobId")

1. **Authentication**

In [ ]:
token = get_oauth_token()

2. **Invocation**

In [ ]:
prompt = """You are an expert career advisor and AI job-matching assistant.
                        I will provide a resume as a pdf. First, extract the text from the resume.
                        Then analyze it to determine the most relevant job roles the candidate is suited for.
                        For each suggested job role, provide:
                        1. Fit Score (0–100) with reasoning
                        2. Matched Skills (from the resume)
                        3. Skill Gaps (skills missing or weak compared to role requirements)
                        4. Targeted Improvement Guidance (practical, specific suggestions to close the skill gaps)
                        Output should be structured in JSON with the following format:
                        {
                        "candidate_summary": "short summary of candidate’s profile",
                        "recommended_roles": [
                        {
                            "role": "job title",
                            "fit_score": number,
                            "matched_skills": ["skill1", "skill2", ...],
                            "skill_gaps": ["missing_skill1", "missing_skill2", ...],
                            "improvement_guidance": "clear, actionable recommendations"
                        }
                        ]
                        }
                        The resume is provided as a pdf """
model_identifier = "anthropic-claude-3-sonnet"
base_64_encoded_data=read_resume(pdf_path='resume_example.pdf')
jobId = invoke_llm(prompt, model_id, token, base_64_encoded_data)

3. **Poll job status** via **Get Job by job ID** until succeeded/failed and Get Data is successful

In [ ]:
if wait_for_job_completion(jobId, token):
            modelops_resp = get_response(jobId, token)
            response={}
            response["body"] = json.dumps(modelops_resp['content'][0]['text'])
        else:
            response = {}

print(response)